# PUBG API Data Extraction & Structuring
This notebook tests the extraction of PUBG match data and structures it into flattened DataFrames suitable for a data pipeline.

In [1]:
import requests
import pandas as pd
from config import api_key
from IPython.display import display

In [ ]:
def get_pubg_samples(api_key, shard='steam'):
    """Fetch a list of sample matches from the PUBG API."""
    url = f"https://api.pubg.com/shards/{shard}/samples"
    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [ ]:
def get_match_details(api_key, match_id, shard='steam'):
    """Fetch detailed information for a specific match."""
    url = f"https://api.pubg.com/shards/{shard}/matches/{match_id}"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [3]:
# 1. Fetch Sample Matches
samples_data = get_pubg_samples(api_key)
if samples_data:
    matches_list = samples_data['data']['relationships']['matches']['data']
    print(f"Found {len(matches_list)} sample matches.")
    first_match_id = matches_list[0]['id']
    print(f"Selected first match ID: {first_match_id}")

Found 868 sample matches.
Selected first match ID: 38df78e0-d98e-43f7-81f0-fd819a084e93


In [4]:
# 2. Fetch Match Details
match_json = get_match_details(api_key, first_match_id)
if match_json:
    print(f"Successfully retrieved details for match {first_match_id}")

Successfully retrieved details for match 38df78e0-d98e-43f7-81f0-fd819a084e93


In [5]:
def parse_match_data(match_json):
    """
    Parses a PUBG match JSON response into separate flattened DataFrames.
    Returns a dictionary of DataFrames: match, participants, rosters, assets.
    """
    # 1. Extract Core Match Data
    match_data = match_json['data']
    df_match = pd.json_normalize(match_data)
    # Flatten attributes and remove prefixes
    df_match.columns = [col.replace('attributes.', '').replace('relationships.', '') for col in df_match.columns]
    
    # 2. Extract Included Objects (Participants, Rosters, Assets)
    included = match_json.get('included', [])
    
    participants = [item for item in included if item['type'] == 'participant']
    rosters = [item for item in included if item['type'] == 'roster']
    assets = [item for item in included if item['type'] == 'asset']
    
    # Flatten Participants
    df_participants = pd.json_normalize(participants)
    if not df_participants.empty:
        df_participants.columns = [col.replace('attributes.', '').replace('stats.', '').replace('relationships.', '') for col in df_participants.columns]
        # Add match_id for reference
        df_participants['match_id'] = match_data['id']
    
    # Flatten Rosters
    df_rosters = pd.json_normalize(rosters)
    if not df_rosters.empty:
        df_rosters.columns = [col.replace('attributes.', '').replace('relationships.', '') for col in df_rosters.columns]
        df_rosters['match_id'] = match_data['id']
    
    # Flatten Assets
    df_assets = pd.json_normalize(assets)
    if not df_assets.empty:
        df_assets.columns = [col.replace('attributes.', '').replace('relationships.', '') for col in df_assets.columns]
        df_assets['match_id'] = match_data['id']
        
    return {
        'match': df_match,
        'participants': df_participants,
        'rosters': df_rosters,
        'assets': df_assets
    }

dfs = parse_match_data(match_json)

In [6]:
print("--- Match Overview ---")
display(dfs['match'].head())

print("\n--- Participants (Player Stats) ---")
display(dfs['participants'].head())

print("\n--- Rosters (Teams) ---")
display(dfs['rosters'].head())

print("\n--- Assets (Telemetry) ---")
display(dfs['assets'].head())

--- Match Overview ---


,type,id,matchType,seasonState,duration,gameMode,isCustomMatch,createdAt,stats,titleId,shardId,tags,mapName,rosters.data,assets.data,links.self,links.schema
0,match,38df78e0-d98e-43f7-81f0-fd819a084e93,official,progress,3894,duo,False,2026-04-17T23:38:06Z,None,bluehole-pubg,steam,None,Savage_Main,"[{'type': 'roster', 'id': '09c0b159-4996-4cee-...","[{'type': 'asset', 'id': 'c4b83c76-3ab9-11f1-8...",https://api.pubg.com/shards/steam/matches/38df...,



--- Participants (Player Stats) ---


,type,id,DBNOs,assists,boosts,damageDealt,deathType,headshotKills,heals,killPlace,...,swimDistance,teamKills,timeSurvived,vehicleDestroys,walkDistance,weaponsAcquired,winPlace,actor,shardId,match_id
0,participant,a7397aef-2667-4e47-8915-0fa3fc6e2d35,1,0,4,72.434130,byzone,0,6,18,...,172.955440,0,3591,0,1770.64030,0,14,,steam,38df78e0-d98e-43f7-81f0-fd819a084e93
1,participant,857c0bb2-3964-4ef9-82f0-ae6e2d362e96,0,0,3,0.000000,suicide,0,0,53,...,4.285492,0,3049,0,1463.81790,0,19,,steam,38df78e0-d98e-43f7-81f0-fd819a084e93
2,participant,373b421b-300e-481a-afa6-60da175f8c91,0,0,1,66.507996,byplayer,0,0,83,...,0.000000,0,2941,0,282.82944,0,42,,steam,38df78e0-d98e-43f7-81f0-fd819a084e93
3,participant,dac1a1af-9843-4b10-b13a-bb96c8e6f1f7,0,0,1,19.358051,byplayer,0,0,84,...,0.000000,0,2941,0,301.06730,0,42,,steam,38df78e0-d98e-43f7-81f0-fd819a084e93
4,participant,e041125c-b9d2-4003-967a-10118a4768c1,0,0,2,124.222280,byplayer,0,1,76,...,20.458303,0,3004,0,569.65970,0,38,,steam,38df78e0-d98e-43f7-81f0-fd819a084e93



--- Rosters (Teams) ---


,type,id,stats.rank,stats.teamId,won,shardId,team.data,participants.data,match_id
0,roster,245d12b7-eabd-4c2a-b93e-471079852836,15,201,false,steam,None,"[{'type': 'participant', 'id': '672b5eda-0dd0-...",38df78e0-d98e-43f7-81f0-fd819a084e93
1,roster,bc7a7789-eb0c-4324-8929-a678a42bc93d,31,245,false,steam,None,"[{'type': 'participant', 'id': 'da49f596-3356-...",38df78e0-d98e-43f7-81f0-fd819a084e93
2,roster,2e017618-83c6-4cc7-bafd-5e5c367c3ad7,7,244,false,steam,None,"[{'type': 'participant', 'id': 'd59f01f1-2a88-...",38df78e0-d98e-43f7-81f0-fd819a084e93
3,roster,38918544-98c8-4c67-ae7a-2cb63cf748a3,30,219,false,steam,None,"[{'type': 'participant', 'id': '52eeab8f-8953-...",38df78e0-d98e-43f7-81f0-fd819a084e93
4,roster,6e8e9c96-8e7e-4291-ab32-ab0de8a11f0f,49,216,false,steam,None,"[{'type': 'participant', 'id': '33640861-0f4e-...",38df78e0-d98e-43f7-81f0-fd819a084e93



--- Assets (Telemetry) ---


,type,id,URL,name,description,createdAt,match_id
0,asset,c4b83c76-3ab9-11f1-8e3e-b2f9d2260867,https://telemetry-cdn.pubg.com/bluehole-pubg/s...,telemetry,,2026-04-18T00:01:36Z,38df78e0-d98e-43f7-81f0-fd819a084e93
